# Model Comparison and Hyperparameter Tuning

In [14]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor,
                              ExtraTreesRegressor,
                              HistGradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [10]:
# Setup paths
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
clean_meta_path = os.path.join(repo_root, "data", "clean", "metadata_clean.parquet")
raw_wave_path = os.path.join(repo_root, "data", "raw", "waveforms.hdf5")

df = pd.read_parquet(clean_meta_path)

# sample_df = df.sample(n=min(1000, len(df)), random_state=42)

trace_matrix = []
matched = []
with h5py.File(raw_wave_path, "r") as hf:
    data_grp = hf["data"]
    for _, row in df.iterrows():
        b_val, t_idx = str(row["bucket"]).strip(), int(row["tindex"])
        b_key = b_val if b_val.startswith("bucket") else f"bucket{b_val}"
        if b_key in data_grp and t_idx < data_grp[b_key].shape[0]:
            tr = np.array(data_grp[b_key][t_idx, :, :300], dtype = np.float32)
            trace_matrix.append(tr) 
            matched.append(row)

X = np.array(trace_matrix)
matched_df = pd.DataFrame(matched).reset_index(drop=True)
y = matched_df["preferred_source_magnitude"].to_numpy(dtype = np.float32)
# X_waveforms: (number of recordings, 300, 3)
# y: corresponding continuous earthquake magnitudes

# Flatten waveforms for traditional ML models
X_ml = X.reshape(X.shape[0], -1)

print("Waveform shape:", X.shape)
print("ML feature shape:", X_ml.shape)
print("Target shape:", y.shape)

# Verify alignment
assert len(X) == len(y)
assert len(X_ml) == len(y)

# Check missing targets
print("Missing magnitudes:", np.isnan(y).sum())

Waveform shape: (18640, 3, 300)
ML feature shape: (18640, 900)
Target shape: (18640,)
Missing magnitudes: 0


In [11]:
# Example: replace with your actual event ID column
groups = df["event_id"].to_numpy()

splitter = GroupShuffleSplit(n_splits = 1,
                             test_size = 0.20,
                             random_state = 42)

train_idx, test_idx = next(splitter.split(X_ml, y, groups = groups))

X_train, X_test = X_ml[train_idx], X_ml[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

groups_train = groups[train_idx]

cv = GroupKFold(n_splits=5)

print("Training recordings:", len(X_train))
print("Testing recordings:", len(X_test))

Training recordings: 14863
Testing recordings: 3777


In [12]:
baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_mae = mean_absolute_error(
    y_test, baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)

baseline_r2 = r2_score(
    y_test, baseline_predictions
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

Baseline MAE: 0.5914019346237183
Baseline RMSE: 0.742776697087966
Baseline R²: -0.0005992650985717773


In [ ]:
# Have not checked or looked through code
# Unfinished

models = {
    "Mean baseline": DummyRegressor(strategy="mean"),

    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42
    ),

    "Neural Network (MLP)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=(128, 64),
            max_iter=300,
            early_stopping=True,
            random_state=42
        ))
    ])
}

results = []
fitted_models = {}

for name, model in models.items():
    start = time.time()

    model.fit(X_train, y_train)

    elapsed = time.time() - start
    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions),
        "Training time (seconds)": elapsed
    })

    fitted_models[name] = model

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("RMSE")

print(results_df.to_string(index=False))


/Users/meredithcarter/ESS_469/MLGEO_AU26_seismology/mlgeo-book/.pixi/envs/default/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.4108848129553735e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


               Model      MAE     RMSE         R2  Training time (seconds)
HistGradientBoosting 0.569867 0.715846   0.070643                 2.049025
       Random Forest 0.570628 0.716448   0.069080                60.648354
         Extra Trees 0.571571 0.718486   0.063775                41.515149
       Mean baseline 0.591402 0.742777  -0.000599                 0.000503
Neural Network (MLP) 0.616861 0.814177  -0.202214                 4.781708
    Ridge Regression 0.706384 2.741340 -12.629168                 0.111297
